In [1]:
## Scenic Env ##

import os
import glob
import gc
import pickle

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc

# pySCENIC imports
from pyscenic.aucell import aucell

%matplotlib inline

/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/deepak/pixi_envs/scenic/.pixi/envs/default/l

In [2]:
# DEFINE PATHS ##

SCENIC_PATH = '/mnt/sdb/scz_meta_analysis_processed/scenic_files/'
FEATHER_RANKINGS_FILE = 'hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather'
DATABASES_GLOB = os.path.join(SCENIC_PATH, FEATHER_RANKINGS_FILE)
MOTIF_ANNOTATIONS_FNAME = os.path.join(SCENIC_PATH, 'motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl')

In [3]:
## SAVE LOAD ##

# # Organoids #
ORGANOIDS_PATH = '/mnt/sdb/scz_meta_analysis_processed/anndata_objs'
organoids_filename = 'integrated_adata_annotated_w_celltypist_scanvi.h5ad'
scz_adata = sc.read_h5ad(os.path.join(ORGANOIDS_PATH, organoids_filename), backed=True)

# TF List ##
TF_LIST = "/home/deepak/datasets/developing_hippocampus/roussos_organoid_snrnaseq/scenic_files/allTFs_hg38.txt"
tf_list = pd.read_csv(TF_LIST, header=None)[0].tolist()

In [4]:
# drop doublets #
droplet_celltype_bool = ((~scz_adata.obs['pred_dbl']) & (scz_adata.obs['subclass_annotations_markers'] != 'Stressed Glia FTL+B2M+GLUL+'))
gc.collect()

0

In [5]:
# Manuscript names mapped to actual AnnData metadata values

manuscripts = {'sebastian': 'Sebastian', 'fernando': 'Fernando', 'rao':'Rao',
                'walsh': 'Walsh', 'notaras': 'Notaras',
                'purcell': 'Purcell', 'khan': 'Khan',
                'shin': 'Shin', 'sawada': 'Sawada'}

In [6]:
# Reload Regulons #

all_regulons = {}

for manuscript_name in manuscripts.keys():

    with open(f"{SCENIC_PATH}/{manuscript_name}_regulons.pkl", "rb") as f:
        all_regulons[manuscript_name] = pickle.load(f)

    print(f"Loaded {manuscript_name} regulons")

Loaded sebastian regulons
Loaded fernando regulons
Loaded rao regulons
Loaded walsh regulons
Loaded notaras regulons
Loaded purcell regulons
Loaded khan regulons
Loaded shin regulons
Loaded sawada regulons


In [10]:
## Calculate AUCs ##

all_auc = {}

for manuscript_name, manuscript_label in manuscripts.items():

    print(f"Processing {manuscript_name}")

    # Load only this manuscript into memory
    adata = scz_adata[(scz_adata.obs['Manuscript'] == manuscript_label) & droplet_celltype_bool].to_memory()
    adata.X = adata.layers["raw_counts"]

    # filter gene space to top 10k HVGs + TFs that show expression in at least 10 cells #
    sc.pp.highly_variable_genes(adata, layer="log1p_norm", n_top_genes=10000, subset=False, inplace=True)
    min_cells = max(10, int(0.01 * adata.n_obs))
    expressed_TFs = adata.var_names[adata.var_names.isin(tf_list) &  ((adata.X > 0).sum(axis=0).A1 >= min_cells)]
    gene_filter = (adata.var["highly_variable"] | adata.var_names.isin(expressed_TFs))
    adata = adata[:, gene_filter].copy()
    
    expr = pd.DataFrame(adata.X.toarray(), index=adata.obs_names, columns=adata.var_names)
    
    print("running AUCell")

    auc_mtx = aucell(expr, all_regulons[manuscript_name], num_workers=1)

    all_auc[manuscript_name] = auc_mtx

    # cleanup
    del expr
    del auc_mtx
    del adata
    gc.collect()

Processing sebastian


/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/scanpy/preprocessing/_highly_variable_genes.py:226: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby("mean_bin")["dispersions"]


running AUCell


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 153/153 [02:50<00:00,  1.11s/it]


Processing fernando


...

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 214/214 [00:11<00:00, 18.25it/s]


Processing sawada


/home/deepak/pixi_envs/scenic/.pixi/envs/default/lib/python3.10/site-packages/scanpy/preprocessing/_highly_variable_genes.py:226: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby("mean_bin")["dispersions"]


running AUCell


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 272/272 [00:00<00:00, 328.32it/s]


In [11]:
## Save AUCs ##

output_file = os.path.join(SCENIC_PATH, "all_manuscript_auc.pkl")

with open(output_file, "wb") as f:
    pickle.dump(all_auc, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved AUC matrices to {output_file}")

Saved AUC matrices to /mnt/sdb/scz_meta_analysis_processed/scenic_files/all_manuscript_auc.pkl
